In [17]:
!nvidia-smi

Thu Apr 30 00:37:32 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   33C    P0             55W /  400W |       6MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [18]:
import sys
import torch

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA build:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("Memory GB:", torch.cuda.get_device_properties(0).total_memory / 1e9)

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
PyTorch: 2.10.0+cu128
CUDA build: 12.8
CUDA available: True
GPU: NVIDIA A100-SXM4-80GB
Memory GB: 85.094825984


In [19]:
from pathlib import Path
import os

repo_dir = Path("/content/modded-nanogpt")
repo_url = "https://github.com/bluepeach1121/modded-nanogpt.git"
branch = "modded-gpt-test"

if repo_dir.exists():
    print("Repo already exists. Updating existing clone.")
    os.chdir(repo_dir)
    !git fetch origin
    !git checkout {branch}
    !git pull
else:
    print("Repo not found. Cloning.")
    %cd /content
    !git clone {repo_url}
    %cd /content/modded-nanogpt
    !git checkout {branch}

print("\nCurrent branch:")
!git branch

print("\nStatus:")
!git status

Repo already exists. Updating existing clone.
remote: Enumerating objects: 11, done.
remote: Counting objects: 100% (11/11), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 6 (delta 5), reused 6 (delta 5), pack-reused 0 (from 0)
Unpacking objects: 100% (6/6), 1.90 KiB | 970.00 KiB/s, done.
From https://github.com/bluepeach1121/modded-nanogpt
   32ecde4..5e3fc58  modded-gpt-test -> origin/modded-gpt-test
Already on 'modded-gpt-test'
Your branch is behind 'origin/modded-gpt-test' by 1 commit, and can be fast-forwarded.
  (use "git pull" to update your local branch)
Updating 32ecde4..5e3fc58
Fast-forward
 records/track_3_optimization/train_gpt_simple.py |   2 +-
 track3_colab_test.ipynb                          | 841 ++---------------------
 2 files changed, 62 insertions(+), 781 deletions(-)

Current branch:
  master
* modded-gpt-test

Status:
On branch modded-gpt-test
Your branch is up to date with 'origin/modded-gpt-test'.

nothing to commit, working tree clean


In [20]:
from pathlib import Path

%cd /content/modded-nanogpt

paths = [
    "data/cached_fineweb10B.py",
    "records/track_3_optimization/train_gpt_simple.py",
    "records/track_3_optimization/README.md",
]

for p in paths:
    path = Path(p)
    print(f"{p}: {path.exists()}")

/content/modded-nanogpt
data/cached_fineweb10B.py: True
records/track_3_optimization/train_gpt_simple.py: True
records/track_3_optimization/README.md: True


In [21]:
%cd /content/modded-nanogpt
!python data/cached_fineweb10B.py 40

/content/modded-nanogpt


In [22]:
from pathlib import Path

data_dir = Path("/content/modded-nanogpt/data/fineweb10B")

print("Data dir exists:", data_dir.exists())
print("Data dir:", data_dir)

if data_dir.exists():
    train_files = sorted(data_dir.glob("fineweb_train_*.bin"))
    val_files = sorted(data_dir.glob("fineweb_val_*.bin"))
    all_bin_files = sorted(data_dir.glob("*.bin"))

    total_gb = sum(f.stat().st_size for f in all_bin_files) / 1e9

    print("\nTrain shards:", len(train_files))
    print("Val shards:", len(val_files))
    print("Total .bin files:", len(all_bin_files))
    print(f"Total size: {total_gb:.3f} GB")

    if train_files:
        print("\nFirst train shard:", train_files[0].name)
        print("Last train shard:", train_files[-1].name)

    if val_files:
        print("Validation shard:", val_files[0].name)

    print("\nLast 10 train shards:")
    for f in train_files[-10:]:
        print(f"{f.name}: {f.stat().st_size / 1e9:.3f} GB")

Data dir exists: True
Data dir: /content/modded-nanogpt/data/fineweb10B

Train shards: 40
Val shards: 1
Total .bin files: 41
Total size: 8.200 GB

First train shard: fineweb_train_000001.bin
Last train shard: fineweb_train_000040.bin
Validation shard: fineweb_val_000000.bin

Last 10 train shards:
fineweb_train_000031.bin: 0.200 GB
fineweb_train_000032.bin: 0.200 GB
fineweb_train_000033.bin: 0.200 GB
fineweb_train_000034.bin: 0.200 GB
fineweb_train_000035.bin: 0.200 GB
fineweb_train_000036.bin: 0.200 GB
fineweb_train_000037.bin: 0.200 GB
fineweb_train_000038.bin: 0.200 GB
fineweb_train_000039.bin: 0.200 GB
fineweb_train_000040.bin: 0.200 GB


In [23]:
from pathlib import Path
import re

script_path = Path("/content/modded-nanogpt/records/track_3_optimization/train_gpt_simple.py")
text = script_path.read_text()

match = re.search(r"train_steps\s*=\s*(\d+)", text)
print("Script exists:", script_path.exists())
print("train_steps:", match.group(1) if match else "not found")

print("\nMuon settings lines:")
for line in text.splitlines():
    if "optimizer2 = Muon" in line or "lr=0.025" in line or "weight_decay=0.0125" in line:
        print(line)

Script exists: True
train_steps: 3500

Muon settings lines:
optimizer2 = Muon([p for p in model.blocks.parameters() if p.ndim >= 2],
                  lr=0.025, weight_decay=0.0125)


In [24]:
%cd /content/modded-nanogpt
!torchrun --standalone --nproc_per_node=1 records/track_3_optimization/train_gpt_simple.py

/content/modded-nanogpt
logs/3f51003a-544e-43ad-ae21-16d4ba6c6cff.txt
step:0/3500 val_loss:10.82583 train_time:0.000s step_avg:0.08ms
step:1/3500 train_time:5.480s step_avg:5479.68ms
step:2/3500 train_time:8.033s step_avg:4016.60ms
step:3/3500 train_time:10.512s step_avg:3503.88ms
step:4/3500 train_time:12.983s step_avg:3245.73ms
step:5/3500 train_time:15.458s step_avg:3091.70ms
step:6/3500 train_time:17.930s step_avg:2988.33ms
step:7/3500 train_time:20.402s step_avg:2914.55ms
step:8/3500 train_time:22.873s step_avg:2859.16ms
step:9/3500 train_time:25.345s step_avg:2816.09ms
step:10/3500 train_time:27.817s step_avg:2781.69ms
step:11/3500 train_time:30.288s step_avg:2753.48ms
step:12/3500 train_time:32.760s step_avg:2730.02ms
step:13/3500 train_time:35.232s step_avg:2710.16ms
step:14/3500 train_time:37.704s step_avg:2693.11ms
step:15/3500 train_time:40.175s step_avg:2678.34ms
step:16/3500 train_time:42.647s step_avg:2665.46ms
step:17/3500 train_time:45.119s step_avg:2654.04ms
step:18/35